# BERT（采用Encoder架构）
$$
Input_{embedding} = Token_{embedding} + Position_{embedding} + Segment_{embedding}
$$
## 掩码语言模型（MLM）
BERT通过掩码语言模型（Masked Language Model, MLM）来预训练。具体做法是随机选择输入序列中的一些词，并将它们替换为一个特殊的掩码标记（[MASK]）。模型的任务是根据上下文预测这些被掩码的词。
## 下一句预测（NSP）
采取二分类任务，输入是两个句子A和B，模型需要判断句子B是否是句子A的下一个句子。这有助于模型理解句子之间的关系。

### 4.1 微调下游任务

（1）文本分类任务：对于文本分类任务（如情感分析、意图识别），主要利用 `[CLS]` 词元的聚合表示能力（见 2.3 节）。

- **输入构造**: 将单个句子或句子对按照 BERT 的要求格式化（添加 `[CLS]` 和 `[SEP]`）。
- **获取表示**: 提取 `[CLS]` 词元对应的最终输出向量。
- **添加分类头**: 在这个向量之上，添加一个简单的全连接层作为分类器，其输出维度等于任务的类别数量。
- **微调**: 在任务数据上，训练这个新添加的分类器，同时对 BERT 模型的参数进行微调（通常使用比预训练时小得多的学习率）。

（2）词元分类任务：对于词元级别的任务（如命名实体识别、分词、词性标注），需要为输入序列中的每一个词元进行分类。

- **输入构造**: 与文本分类类似，将序列格式化。
- **获取表示**: 将输入送入 BERT 模型，但这次提取 **所有词元** 对应的最终输出向量序列。输出的形状为 `(batch_size, sequence_length, hidden_size)`。
- **添加分类头**: 在这个序列之上，添加一个全连接层（可以看作是在时间维度上共享权重），将每个词元的 `hidden_size` 维向量映射到类别数量的维度。
- **微调**: 在任务数据上进行端到端的训练和微调。

In [ ]:
import os
# 必须在最开头设置，且值不能有空格！
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

# 再导入其他模块
import torch
from transformers import AutoTokenizer, AutoModel
from huggingface_hub import hf_hub_url

print("实际请求的 URL 示例:")
print(hf_hub_url("bert-base-chinese", "config.json"))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = r"d:\deeplearning\bert-base-chinese-local"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

text = ["我来自中国", "我喜欢自然语言处理"]
inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)

for key, value in inputs.items():
    print(f"{key}: {value.shape}")

with torch.no_grad():
    outputs = model(**inputs)

last_hidden_states = outputs.last_hidden_state
sentence_features_pooler = getattr(outputs, "pooler_output", None)
sentence_features = last_hidden_states[:, 0, :]
first_sentence_tokens = last_hidden_states[0, 1:6, :]

print("\n--- 特征提取结果 ---")
print(f"句子特征 shape: {sentence_features.shape}")
if sentence_features_pooler is not None:
    print(f"pooler_output shape: {sentence_features_pooler.shape}")
print(f"第一个句子的词元特征 shape: {first_sentence_tokens.shape}")

print(model)

实际请求的 URL 示例:
https://huggingface.co/bert-base-chinese/resolve/main/config.json


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1002.34it/s, Materializing param=pooler.dense.weight]                              
BertModel LOAD REPORT from: d:\deeplearning\bert-base-chinese-local
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


input_ids: torch.Size([2, 11])
token_type_ids: torch.Size([2, 11])
attention_mask: torch.Size([2, 11])

--- 特征提取结果 ---
句子特征 shape: torch.Size([2, 768])
pooler_output shape: torch.Size([2, 768])
第一个句子的词元特征 shape: torch.Size([5, 768])
BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(21128, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (

: 

In [14]:
import requests
import certifi

url = "https://hf-mirror.com/bert-base-chinese/resolve/main/config.json"

try:
    resp = requests.get(url, verify=certifi.where())
    print("状态码:", resp.status_code)
    if resp.status_code == 200:
        print("✅ 可成功下载 config.json")
        print("前200字符内容预览:")
        print(resp.text[:200])
    else:
        print("❌ 返回非200状态码，可能被限流或拦截")
        print("响应内容:")
        print(resp.text[:500])
except Exception as e:
    print("❌ 请求失败:", e)

状态码: 200
✅ 可成功下载 config.json
前200字符内容预览:
{
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "directionality": "bidi",
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "init
